## Transformer Model: DistilBERT Sentiment Classification

This notebook trains a transformer-based model for sentiment
classification using the HuggingFace Transformers library.

Model used:

DistilBERT (distilbert-base-uncased)

DistilBERT is a lightweight version of BERT that retains most
of the performance while requiring less computational resources.

Steps performed:

1. Load processed training and testing datasets
2. Convert data to HuggingFace dataset format
3. Tokenize review text using DistilBERT tokenizer
4. Fine-tune the transformer model
5. Save the trained model for later evaluation

In [3]:
#Import required libraries

import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

#Device check
#Check if GPU is available
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

#Load processed data
train_df = pd.read_csv("../data/train.csv")
test_df  = pd.read_csv("../data/test.csv")

#Print train dataset
train_df.head()

CUDA available: True
GPU: NVIDIA GeForce GTX 1650


,Score,Text,sentiment,label
0,1,Eukanuba is a well-respected name in pet foods...,negative,0
1,1,We tried this with our 10-month-old son and he...,negative,0
2,3,I thought I was getting the standard package t...,neutral,1
3,3,"This food may be good for my cats, but two of ...",neutral,1
4,5,This is my favorite coffee outside of Italy. I...,positive,2


In [4]:
#Dataset Preparation 
#Tokenization
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#Convert pandas dataframe to HuggingFace dataset format
train_dataset = Dataset.from_pandas(train_df[["Text", "label"]])
test_dataset  = Dataset.from_pandas(test_df[["Text", "label"]])

def tokenize_function(examples):
    """
    Tokenize review text for DistilBERT input
    Parameter: 
    Examples: dict
        Dictionary containing reveiw text
    Returns:
    dict
        Tokenized input & attention masks
    """
    
    return tokenizer(
        examples["Text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
#Apply tokenization to entire dataset
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

#Remove raw text column - model uses tokens
train_dataset = train_dataset.remove_columns(["Text"])
test_dataset  = test_dataset.remove_columns(["Text"])

#Convert to PyTorch tensors
train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/102336 [00:00<?, ? examples/s]

Map:   0%|          | 0/25584 [00:00<?, ? examples/s]

In [5]:
#Model Setup

#load pretrained DistilBERT model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3 #Adjust output layer for 3 sentiment classes
)

def compute_metrics(eval_pred) -> dict:
    """ 
    Compute evaluation metrics during triaing
    Parameters:
    eval_pred : tuple
        Model prediciton and true lables
    Returns:
    dict
        Accuracy, precision, recall and F1
    """
    
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

#Configure training parameters
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=200,
)

#Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.522317,0.494191,0.797178,0.799829,0.816020,0.797178
2,0.360993,0.461408,0.826689,0.827042,0.827529,0.826689


TrainOutput(global_step=12792, training_loss=0.47531674488251324, metrics={'train_runtime': 5633.8657, 'train_samples_per_second': 36.329, 'train_steps_per_second': 2.271, 'total_flos': 6778212732076032.0, 'train_loss': 0.47531674488251324, 'epoch': 2.0})

In [18]:
#Save model
trainer.save_model("../models/distilbert_3class_balanced_v1")
tokenizer.save_pretrained("../models/distilbert_3class_balanced_v1")
print("model saved")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model saved


In [20]:
#Final Evaluation
results = trainer.evaluate()
print(results)

{'eval_loss': 0.4614076018333435, 'eval_accuracy': 0.826688555347092, 'eval_f1': 0.8270424927509242, 'eval_precision': 0.8275294696250615, 'eval_recall': 0.826688555347092, 'eval_runtime': 220.867, 'eval_samples_per_second': 115.834, 'eval_steps_per_second': 7.24, 'epoch': 2.0}
